# 08.06 · Streaming in depth

**Module 08 · LangGraph Production Features** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 08.01–08.05; Lesson 01.05 (streaming a model). |
| 🔑 Keys | none (optional: `GROQ_API_KEY` for live tokens in Part 5) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Choose** the right stream mode: `values`, `updates`, `messages`, `custom`, `checkpoints`, `tasks`, `debug`.
2. **Read** the v2 stream format (`{"type", "ns", "data"}`), combine modes, and include subgraphs.
3. **Stream** LLM tokens from inside nodes, filtered by node, and your own progress events.
4. **Show** a stream in a notebook with a live-updating display, and know the experimental v3 `stream_events` API.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Streaming](https://docs.langchain.com/oss/python/langgraph/streaming) | Every stream mode, `version="v2"`, `subgraphs=True`, the custom writer, filtering tokens |
| [Event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming) | `stream_events(version="v3")`: typed projections (`.messages`, `.values`, `.output`, ...) |
| [Streaming (LangChain)](https://docs.langchain.com/oss/python/langchain/streaming) | The same modes for `create_agent` agents (Module 02) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langgraph>=1.2.14,<2" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langgraph", "langchain-core", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/08-langgraph-production-features/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY',)
# Setup 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma (Project 08 can reuse Module 06): no usage telemetry.
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)

---
## Part 1 · Seven modes on one tiny graph

A two-node graph: `research` emits a custom progress event, `answer` calls a (scripted) chat model. `ScriptedChatModel` streams its reply word by word, like a real model.

In [ ]:
# 🧪 Offline: the test graph.
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer
from langgraph.graph import START, MessagesState, StateGraph
from mastery.testing import ScriptedChatModel

model = ScriptedChatModel(responses=[AIMessage("Cold weather cuts battery life by about forty percent.")], repeat_last=True)


def research(state: MessagesState) -> dict:
    writer = get_stream_writer()                        # your own events, any JSON-like value
    for i, source in enumerate(["faq-battery.md", "battery-handling.pdf"], 1):
        writer({"progress": f"read {source}", "done": i, "total": 2})
    return {}


def answer(state: MessagesState) -> dict:
    return {"messages": [model.invoke(state["messages"])]}


graph = (StateGraph(MessagesState).add_node("research", research).add_node("answer", answer)
         .add_edge(START, "research").add_edge("research", "answer").compile(checkpointer=InMemorySaver()))
QUESTION = {"messages": [HumanMessage("How does cold affect the battery?")]}

In [ ]:
# 🧪 Offline: count the events each mode produces.
MODES = ["values", "updates", "messages", "custom", "checkpoints", "tasks", "debug"]
for i, mode in enumerate(MODES):
    events = list(graph.stream(QUESTION, {"configurable": {"thread_id": f"modes-{i}"}}, stream_mode=mode, version="v2"))
    first = events[0]["data"]
    preview = type(first).__name__ if not isinstance(first, dict) else "dict keys " + str(list(first)[:4])
    print(f"{mode:<12} {len(events):>2} events   first: {preview}")

**You should now see**:

| Mode | Events | What each event is | Use it for |
|---|---|---|---|
| `values` | 2 | The **full state** after each step | Simple UIs, debugging |
| `updates` | 2 | `{node: its update}` per node | Progress ("✅ research done") |
| `messages` | 9 | `(token chunk, metadata)` from LLM calls in nodes | Typing effect in chat UIs |
| `custom` | 2 | Whatever you `writer(...)` | Your own progress, partial results |
| `checkpoints` | 4 | A checkpoint was saved | Persistence-aware UIs |
| `tasks` | 4 | A task started / finished (with result or error) | Monitoring |
| `debug` | 8 | Checkpoints + tasks, with timestamps | Deep debugging (it's chatty) |

### What just happened?

One run, seven views of it. Every mode reports the **same** execution at a different level: whole states, per-node updates, LLM tokens, your own events, or the runtime's bookkeeping (checkpoints, tasks). You choose per consumer: a chat UI wants `messages`, a progress bar `updates` or `custom`, an ops dashboard `tasks`.

Exact counts depend on the graph; these are for 2 nodes, a 9-word reply and a checkpointer (`checkpoints`/`tasks`/`debug` need one or are empty). `values` has 2, not 3: `research` returns `{}`, so the state doesn't change after it and no new values event is sent.

---
## Part 2 · The v2 format, several modes at once, subgraphs

With `version="v2"` every event is a dict `{"type": mode, "ns": namespace, "data": payload}`, whatever modes you ask for. Pass a list of modes to get them interleaved, in order.

In [ ]:
# 🧪 Offline: progress + node updates + tokens in ONE stream.
for part in graph.stream(QUESTION, {"configurable": {"thread_id": "multi"}},
                         stream_mode=["custom", "updates", "messages"], version="v2"):
    if part["type"] == "custom":
        print(f"📖 {part['data']['progress']} ({part['data']['done']}/{part['data']['total']})")
    elif part["type"] == "updates":
        print("✅", ", ".join(part["data"]), "done")
    elif part["type"] == "messages":
        chunk, meta = part["data"]
        print(chunk.text, end="", flush=True)
print()

**You should now see** `📖 read faq-battery.md (1/2)`, `📖 read battery-handling.pdf (2/2)`, `✅ research done`, the answer typed out word by word, then `✅ answer done` on the same line, right after the last word (a node's update arrives when the node returns; add a `print()` before it in a real UI).

> Without `version="v2"`, a single mode yields bare payloads and a list of modes yields `(mode, payload)` tuples (and `(ns, mode, payload)` with subgraphs). v2 is one shape for everything; the course uses it.

In [ ]:
# 🧪 Offline: only tokens from ONE node, using the metadata that comes with each chunk.
tokens = []
for part in graph.stream(QUESTION, {"configurable": {"thread_id": "filter"}}, stream_mode="messages", version="v2"):
    chunk, meta = part["data"]
    if meta["langgraph_node"] == "answer":
        tokens.append(chunk.text)
print(len(tokens), "tokens from 'answer':", "".join(tokens))
print("metadata keys:", sorted(k for k in meta if k.startswith("langgraph_")))

**You should now see** `9 tokens from 'answer': Cold weather cuts battery life by about forty percent.` and metadata keys such as `langgraph_node`, `langgraph_step`, `langgraph_checkpoint_ns`, `langgraph_triggers`.

With several model calls (a planner and a writer), filter on `langgraph_node`, or tag a model (`model.with_config(tags=["writer"])`) and filter on `meta["tags"]`. Add `subgraphs=True` to get tokens from inside subgraphs; `part["ns"]` tells you which one (Lesson 08.05).

---
## Part 3 · A live-updating display

Printing tokens works in a terminal. In a notebook you can do better: keep **one** output area and update it in place with `display(..., display_id=True)`. Here: a progress line plus the answer growing as tokens arrive.

In [ ]:
# 🧪 Offline: one Markdown area, updated on every event.
import time

from IPython.display import Markdown, display

view = display(Markdown("⏳ waiting…"), display_id=True)
progress, text = "", ""
for part in graph.stream(QUESTION, {"configurable": {"thread_id": "live-view"}},
                         stream_mode=["custom", "messages"], version="v2"):
    if part["type"] == "custom":
        progress = f"📖 {part['data']['progress']} ({part['data']['done']}/{part['data']['total']})"
    else:
        text += part["data"][0].text
    view.update(Markdown(f"{progress}\n\n**Answer:** {text}▌"))
    time.sleep(0.05)                                   # only so you can see it move
view.update(Markdown(f"✅ done\n\n**Answer:** {text}"))
print("final answer has", len(text.split()), "words")

**You should now see** one box that changes as you watch (progress, then the answer growing with a `▌` cursor) and ends as `✅ done` with the full answer, plus `final answer has 9 words`.

> A re-opened notebook shows only the **last** state of the box (outputs are saved once). For a real app UI, Module 12 streams the same events over HTTP from `langgraph dev`/LangSmith Deployment.

---
## Part 4 · Experimental: `stream_events(version="v3")`

LangGraph 1.2 adds a typed event-streaming API. Instead of parsing dicts, you get **projections**: `.messages` (one stream object per LLM message, with `.text` to iterate tokens), `.values`, `.output`, `.interrupts`, `.subgraphs`. It is marked **experimental** (it emits a `LangChainBetaWarning`): The docs now call event streaming the recommended in-process model; the runtime still marks it experimental, so pin your langgraph version if you adopt it, and keep `stream(version="v2")` where you need stability.

In [ ]:
# 🧪 Offline: the same run through the v3 projections.
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    run = graph.stream_events(QUESTION, {"configurable": {"thread_id": "v3"}}, version="v3")
    for message in run.messages:                        # one item per LLM message
        print(f"[{message.node}]", " | ".join(message.text))
    print("output:", run.output["messages"][-1].text)
print("warning:", sorted({str(w.message) for w in caught}))

**You should now see** `[answer] Cold  | weather  | cuts  | ...` (each token), `output: Cold weather cuts battery life by about forty percent.` and `warning: ['The v3 streaming protocol on Pregel is experimental.']`.

`astream_events(version="v3")` is the async version. Check the [Event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming) page before using it: experimental APIs can change in minor releases.

---
## Part 5 · The Project 08 agent, streamed

Project 08 wants **node updates and tokens**. `mastery.graphs.hitl.run_until_pause` already streams `updates`; here we combine `updates` + `messages` + `subgraphs` by hand on the research graph.

In [ ]:
# 🧪 Offline: progress lines for every node (parent and research subgraph), tokens from `draft`.
from mastery.graphs.research import build_research_graph, corpus_docs_search, offline_models, offline_web_search
from mastery.graphs.state import memory_checkpointer


async def stream_agent(agent, question, thread_id):
    cfg = {"configurable": {"thread_id": thread_id}}
    drafted = 0
    async for part in agent.astream({"question": question}, cfg, stream_mode=["updates", "messages"],
                                    subgraphs=True, version="v2"):
        where = " > ".join(ns.split(":")[0] for ns in part["ns"])
        if part["type"] == "updates":
            for node in part["data"]:
                print(f"✅ {where + ' > ' if where else ''}{node}")
        elif part["data"][1]["langgraph_node"] == "draft":
            drafted += len(part["data"][0].text)
    print(f"   (draft streamed {drafted} characters)")


agent = build_research_graph(offline_models(), web_search=offline_web_search(), docs_search=corpus_docs_search(),
                             checkpointer=memory_checkpointer())
await stream_agent(agent, "How do I keep the battery healthy in winter?", "stream-1")

**You should now see** `✅ clarify_question`, `✅ plan`, `✅ research > search_docs`, `✅ research > search_web`, `✅ research`, `✅ draft`, `✅ __interrupt__` (the approval pause) and `(draft streamed 317 characters)`: the offline writer is a `ScriptedChatModel`, so it streams like a real model. The clarifier and planner are scripted structured-output models whose output isn't text, so they add nothing to the count.

> 💸 Calls `gpt-oss-20b` 3 times (clarify, plan, draft), a few thousand tokens of your free-tier quota. Without `GROQ_API_KEY` it prints `⏭️ Skipped`.

In [ ]:
# 💸 Live: the same stream with gpt-oss-20b: now `draft` streams tokens.
if HAVE_GROQ_KEY:
    from mastery.models import get_model

    llm = get_model("fast", temperature=0, reasoning_effort="low")
    live = build_research_graph({"clarifier": llm, "planner": llm, "writer": llm}, web_search=offline_web_search(),
                                docs_search=corpus_docs_search(), checkpointer=memory_checkpointer())
    await stream_agent(live, "How do I keep the battery healthy in winter?", "stream-live")
else:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")

**You should now see** (with a key) the same progress lines (unless the model asks a clarifying question first: then the stream stops at that `✅ __interrupt__` after `clarify_question`) and `(draft streamed N characters)` with N typically in the hundreds or more: the writer's tokens. With real models the structured-output calls (clarify, plan) stream too, as tool-call chunks with empty `content`; filter on `langgraph_node` to show only what a user should read. Without a key: `⏭️ Skipped`.

---
## ⚠️ Pitfalls and troubleshooting (Windows Git Bash included)

| Symptom | Cause | Fix |
|---|---|---|
| `messages` mode yields nothing | The node doesn't call a LangChain chat model (or the model doesn't stream) | Use a chat model in the node; plain functions produce no tokens |
| Tokens from the planner leak into the chat UI | Unfiltered `messages` | Filter on `meta["langgraph_node"]` or tags |
| `ValueError: too many values to unpack` | Mixing v1 tuple shapes with v2 | Always `version="v2"` and read `part["type"]`, `part["data"]` |
| `checkpoints`/`tasks` mode is empty | No checkpointer | Compile with one |
| Garbled `°` or emoji in the Git Bash console | cp1252 console | `export PYTHONUTF8=1` (JupyterLab is fine) |
| The notebook freezes on a long stream | Thousands of `display` updates | Update every N tokens, or use `print(..., end="")` |

## ⚠️ Production notes

> ⚠️ **Production note: stream what the user needs, log the rest.**
> `updates` + filtered `messages` + `custom` progress is the usual UI mix. `debug` is for you, not for users: it's verbose and can include internal state.

> ⚠️ **Production note: streams can stop half-way.**
> **In-process** (this notebook), closing the stream generator stops the run at that point: nothing after it executes. On the **Agent Server** (Module 12), a client disconnect ends the stream but, by default (`on_disconnect="continue"`), not the run. Clients should handle partial output, and the server keeps the thread's checkpoints, so the client can rejoin (`runs.join_stream`) or fetch the final state (`threads.get_state`) afterwards.

## 🏋️ Exercises

1. **(Easy)** Stream Part 1's graph with `stream_mode="updates"` and **without** `version="v2"`. What's the shape of each event?
2. **(Medium)** Make the `research` node emit a custom event with **partial results** (`{"found": source}`) and show them in the live display under the progress line.
3. **(Medium)** In Part 5, count how many events of each `type` the offline agent produces.
4. **(Stretch)** Use the v3 API's `.values` projection to print each intermediate state's message count.

<details>
<summary><b>Show solutions</b></summary>

**1.** Bare dicts `{node: update}` (no `type`/`ns` wrapper). With a list of modes you'd get `(mode, data)` tuples.

**2.**
```python
found = []
...
if part["type"] == "custom" and "found" in part["data"]:
    found.append(part["data"]["found"])
view.update(Markdown(progress + "\n\nFound: " + ", ".join(found) + f"\n\n**Answer:** {text}▌"))
```

**3.**
```python
from collections import Counter
counts = Counter()
async for part in agent.astream({"question": "..."}, {"configurable": {"thread_id": "count"}},
                                stream_mode=["updates", "messages"], subgraphs=True, version="v2"):
    counts[part["type"]] += 1
print(counts)   # 'updates' and 'messages' (the draft's tokens)
```

**4.**
```python
run = graph.stream_events(QUESTION, {"configurable": {"thread_id": "v3-values"}}, version="v3")
for state in run.values:
    print(len(state["messages"]))
```
</details>

## ✅ Checkpoint

- [ ] I can name the seven stream modes and pick one for a given UI.
- [ ] I read v2 events (`type`, `ns`, `data`) and combined modes in one stream.
- [ ] I streamed tokens from one node only and my own progress events.
- [ ] I built a live-updating display and tried the experimental v3 API.
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Docs drill (10 min)

How do you stop one model's tokens from appearing in `stream_mode="messages"`? Find `TAG_NOSTREAM` in the source, then use it on the planner.

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. `from langgraph.constants import TAG_NOSTREAM; TAG_NOSTREAM` (the tag value).
2. `langgraph/pregel/_messages.py`, search `TAG_NOSTREAM`: the messages stream handler skips runs that carry this tag.
3. Apply: `planner_model.with_config(tags=[TAG_NOSTREAM])`, run the stream again, and check that only the writer's tokens arrive.

</details>

## 🔎 Research checkpoint

Read [Event streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming) and note in [`docs/research-log.md`](../../docs/research-log.md): which projections exist, how custom transformers (`StreamTransformer`) work, and whether v3 is still marked experimental in the newest langgraph release.

## Summary

- `values` = full state, `updates` = per node, `messages` = tokens, `custom` = yours; `checkpoints`/`tasks`/`debug` for monitoring.
- `version="v2"`: every event is `{"type", "ns", "data"}`; combine modes with a list; `subgraphs=True` adds child events.
- Filter tokens with `meta["langgraph_node"]`; emit progress with `get_stream_writer()`.
- `display(..., display_id=True).update(...)` for live notebook output; `stream_events(version="v3")` is the experimental typed API.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 08.07 · Long-term memory store inside graphs** → [`08-07-stores-in-graphs.ipynb`](08-07-stores-in-graphs.ipynb).

[Module 08 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)